# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This is the deliverable the previous five cards were building toward: a ranked queue a human can
work, with a stated boundary around what it is allowed to decide.

**The operating decision, made before anything else, because everything follows from it: the queue
is worked 500–1000 pages per cycle, so the random forest ships.** ML-08 deliberately left this open
— logistic regression matches the forest in the top 50 (0.788 vs 0.776) and the forest only pulls
ahead with depth. ML-09 settled it: at precision@1000 the forest led the ML-07 rule on **5 of 5**
held-out client folds; at precision@50, on **4 of 5**. Depth is where the advantage is consistent,
so depth is what the queue is built for.

That decision has a price, and this notebook pays it in the open rather than in a footnote: **a
forest cannot explain a row.** Section 1 measures exactly how much of the queue the readable ML-07
rule can justify, finds it is only about half, and closes the rest with codes derived from the
model's own printed behaviour — kept in a separate column so a reviewer always knows which scorer
is talking.

Skill loaded: `skills/writing-honest-claims`, with the queue mechanics in
`work/scripts/action_playbook.py`.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### How a page gets its position

1. **Score.** Random forest probability of `is_declining_label`, **out of fold** — every page is
   scored by a forest fitted on the four client folds that exclude its own client. A forest fitted
   on all 30,000 rows would be scoring its own training data, and the queue's quality numbers would
   be the naive-split figure ML-09 retired.
2. **Eligibility.** Pages with `impressions_prev_30d == 0` are **held out of the queue entirely**,
   not ranked low. There are 3,388 of them, they are exactly the `new` and `flat` pages, and all
   3,388 are labelled not-declining by construction — the label needs a prior window and they do
   not have one. Ranking them would be an artefact of the label definition, not a prediction.
3. **Rank.** By model score, ties broken by `impressions_90d`, so where the model is indifferent the
   more visible page is reviewed first.
4. **Explain.** ML-07 reason codes where the rule fires; model-derived codes where it does not.

### The explainability finding

This is the part I did not expect and it is the most useful thing in the card.

**The ML-07 rule can justify only 55.6% of the queue.** Not because the rule is broken, but because
the two scorers are looking at different things. The rule fires on thinness, staleness, and weak
CTR. The forest ranks on sustained visibility and age — ML-08 measured
`days_with_impressions`, `log_impressions_90d`, `avg_position` and `content_age_days` at roughly 45%
of total importance, and `days_since_last_update`, the rule's single heaviest weight, at 12th place.
**The rule has no reason code for the pattern the model actually ranks on.**

Shipping 444 unexplained rows was not acceptable, so the gap is closed with three model-derived
codes in their own column. The first one is the **depth-2 tree's entire learned rule**, converted
out of standardised units back into readable ones:

> `days_with_impressions > 12` **and** `content_age_days <= 373` → at risk

ML-08 printed that tree specifically so it could be quoted somewhere useful. This is that place.

### An honest limitation of those codes

The cell below shows `consistently_visible_not_yet_old` and `model_high_risk` firing on **1000 of
1000** delivered rows. A code that fires on everything explains the queue *as a population* — every
page in the top 1000 is a consistently-visible, not-yet-old page the model scores above 0.70 — but
it says nothing about why row 12 sits above row 400. **The within-queue ordering is not explainable
from any code set I have.** That is stated here, repeated in section 2's limits, and it is the
honest reason the queue is delivered as a prioritised set to work through rather than as a strict
ordering to defend page by page.

In [1]:
# Section 1 - build the queue and read its shape.
import json
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "work" / "scripts"))

from action_playbook import OUT_CSV, OUT_JSON, run  # noqa: E402

REBUILD_QUEUE = False  # set True to regenerate from scratch (~1 min)

if REBUILD_QUEUE or not OUT_JSON.exists():
    playbook = run()
    OUT_JSON.write_text(json.dumps(playbook, indent=2), encoding="utf-8")
else:
    playbook = json.loads(OUT_JSON.read_text(encoding="utf-8"))

decision = playbook["operating_decision"]
corpus = playbook["corpus"]
print(f"shipped model : {decision['shipped_model']}   queue depth: {decision['queue_depth']}")
print(f"scoring       : {decision['scoring']}")
print(f"corpus        : {corpus['rows']:,} rows, {corpus['clients']} clients, "
      f"base rate {corpus['base_rate']}")
print(f"held out      : {corpus['held_out_no_prior_window']:,} pages with no prior 30-day window "
      f"({corpus['held_out_declining_count']} of them declining - by construction, none)")
print(f"eligible      : {corpus['eligible_for_queue']:,} pages ranked")

print("\n--- queue quality at the depths a cycle might reach ---")
print(pd.DataFrame(playbook["queue_quality"]).T.to_string())

print("\n--- explainability ---")
coverage = pd.DataFrame(playbook["explainability"]["coverage_by_depth"]).T
print(coverage.to_string())
print(f"\nrule codes cover      : {playbook['explainability']['delivered_rows_with_a_rule_code']} rows")
print(f"model codes needed for: {playbook['explainability']['delivered_rows_needing_a_model_code']} rows")
print(f"left unexplained      : {playbook['explainability']['delivered_rows_unexplained']} rows")
print(f"tree rule quoted      : {playbook['explainability']['depth_2_tree_rule_in_readable_units']}")

print("\naction mix across the delivered 1000:")
for action, count in playbook["explainability"]["action_mix"].items():
    print(f"  {action:<32} {count:>4}")

print("\nmodel reason codes (note the two that fire on everything):")
for codeword, count in playbook["explainability"]["model_reason_mix"].items():
    print(f"  {codeword:<34} {count:>4} / {decision['queue_depth']}")

queue = pd.read_csv(OUT_CSV)
print(f"\n--- top 8 of the delivered queue ({len(queue):,} rows, {OUT_CSV.name}) ---")
print(queue.head(8)[[
    "queue_rank", "model_score", "confidence", "suggested_action",
    "explanation_source", "impressions_90d", "avg_position", "ctr", "content_age_days",
]].to_string(index=False))

shipped model : random_forest   queue depth: 1000
scoring       : out-of-fold; every page scored by a forest that never saw its client
corpus        : 30,000 rows, 32 clients, base rate 0.5421
held out      : 3,388 pages with no prior 30-day window (0 of them declining - by construction, none)
eligible      : 26,612 pages ranked

--- queue quality at the depths a cycle might reach ---
        precision  lift_vs_base_rate  declining_pages_found  wasted_reviews
p@100       0.770             1.4205                   77.0            23.0
p@250       0.804             1.4832                  201.0            49.0
p@500       0.782             1.4426                  391.0           109.0
p@1000      0.778             1.4352                  778.0           222.0

--- explainability ---
          rule_only_codes  any_code
top_100             0.540       1.0
top_250             0.588       1.0
top_500             0.566       1.0
top_1000            0.556       1.0

rule codes cover      : 556

**A denominator note, so two numbers in this repo are not mistaken for each other.**

The delivered queue reaches **precision@1000 = 0.778**. ML-09 reported **0.726 ± 0.054** for the
same model at the same metric name. These are not the same measurement and the queue has not
improved:

- ML-09's figure is the top 1,000 rows *within each fold* of roughly 5,300 test rows — averaged
  across five folds. It is the number that answers "how does this model generalise to an unseen
  client?"
- This figure is the top 1,000 of **26,612 eligible pages**. A far more selective cut, so a higher
  precision is expected arithmetically.

The generalisation claim stays **0.726 ± 0.054**. The 0.778 describes this specific delivered
artefact and travels nowhere else. Mixing them would be the exact move section 4 of ML-09 retired.

## 1b. Archetype -> action mapping

*The queue says which page next. It does not yet say what to do when you get there.*

Every one of the 1,000 delivered rows currently carries the same instruction: **look at this page.**
The `suggested_action` column inherited from ML-07 helps where the rule fired, but ML-09 and
section 1 both established that the rule has nothing to say about 44.4% of the queue, and the
model-derived codes that cover the gap fire on 100% of rows — they explain the population, not the
page. So the delivered artefact is a priority order with a single undifferentiated action attached
to it.

This section adds the missing layer: **one mutually exclusive archetype per page, each implying a
different kind of edit.**

### The mapping

| Archetype | Condition (observable page state) | Action | The reasoning |
|---|---|---|---|
| `snippet_gap` | page-one position, 500+ impressions, CTR below its **position peers** | `rewrite_title_and_meta` | It ranks and it is seen. The listing is losing the click, not the body. |
| `thin_with_demand` | word count 1–1,199 with 250+ impressions | `expand_depth` | Measured thin against measured demand. |
| `stale_authority` | page-one position, 500+ impressions, untouched 90+ days | `refresh_facts_and_dates` | A strong page going stale. Defend it before it slips. |
| `deep_but_substantial` | position past 20 despite 1,200+ words | `reassess_intent_or_consolidate` | Length is not the problem. The target query might be. |
| `weak_engagement` | 30+ sessions with engagement or scroll under 30% | `review_intro_and_layout` | They arrive and leave. The defect is above the fold. |
| `low_demand` | under 250 impressions | `no_content_edit_review_targeting` | **Do not edit.** The page may be fine; the demand is not there. |
| `insufficient_page_data` | word count missing | `fix_the_record_first` | The thin/substantial tests could not run. Fix the data first. |
| `no_clear_defect` | none of the above | `manual_diagnosis` | Reported, not hidden. |

### Three design decisions worth stating

**CTR is compared to position peers, not to a flat threshold.** ML-07 uses `ctr < 0.5`. But CTR
falls with position, so a flat cut calls every deep page a click problem. `snippet_gap` compares a
page against the median CTR of its own position band, computed only over pages with 500+
impressions. That last restriction matters: at corpus level the top-3 band's median CTR reads
**0.00**, which looks like page-one pages go unclicked — until you notice those rows have a median
of **3 impressions**. Position is meaningless without visibility, and the peer baseline is built
where it is not.

**Order is load-bearing.** 8,713 of the 26,612 eligible pages satisfy more than one definition, so
first-match-wins is doing real work. The order runs cheapest-and-most-reversible first: a page that
is both a snippet problem and a consolidation candidate gets offered the title rewrite, because
retitling is a smaller act than merging two pages and it is easier to undo.

**Word-count-free tests run first.** `insufficient_page_data` sits second from last, not first, so
a page with a missing word count is still diagnosed on whatever evidence does exist. It only falls
through when nothing word-count-free explained it either.

### What this is not

An archetype is a **triage aid, not evidence.** There is no refresh outcome anywhere in this data,
so nothing here measures whether `expand_depth` recovers traffic. The archetype selects the review
question a human should ask. It does not answer it, and it does not authorise an edit.

In [2]:
# Section 1b - the archetype mix, in the corpus and in what actually ships.
archetypes = playbook["archetypes"]
buckets = archetypes["buckets"]

print(f"eligible pages : {archetypes['eligible_pages']:,}   base rate: {archetypes['eligible_base_rate']:.3f}")
print(f"pages matching >1 definition : {archetypes['order_matters_for_pages']:,} "
      f"({archetypes['order_matters_for_pages'] / archetypes['eligible_pages']:.0%}) "
      "-> priority order is load-bearing")

mix = pd.DataFrame([
    {
        "archetype": name,
        "action": block["action"],
        "eligible": block["eligible_pages"],
        "elig %": block["eligible_share"],
        "in queue": block["queue_pages"],
        "queue %": block["queue_share"],
        "skew": round(block["queue_share"] / block["eligible_share"], 2) if block["eligible_share"] else 0.0,
        "declining": block["declining_rate"],
    }
    for name, block in buckets.items()
])
print("\n--- what the model actually selects, by archetype ---")
print(mix.to_string(index=False))

assert mix["eligible"].sum() == archetypes["eligible_pages"], "archetypes must partition the corpus"
assert mix["in queue"].sum() == len(queue), "every delivered row must carry exactly one archetype"

# The queue's shape is a claim about the model's behaviour, so state it as one.
top3 = mix.nlargest(3, "in queue")
print(f"\ntop 3 archetypes carry {top3['queue %'].sum():.0%} of the delivered queue:")
print("  " + ", ".join(f"{r['archetype']} ({r['queue %']:.0%})" for _, r in top3.iterrows()))

deselected = mix[mix["skew"] < 0.6].sort_values("skew")
print("\n--- what the model de-selects (skew < 0.6x its corpus share) ---")
print(deselected[["archetype", "action", "elig %", "queue %", "skew"]].to_string(index=False))
print("-> two of these are the model behaving well, and it was told to do neither:")
print("   insufficient_page_data at 0.02x - pages whose record is too incomplete to diagnose;")
print("   low_demand at 0.52x - pages with no demand for an edit to serve.")
print("   Two are a limitation, and the more useful half of this table:")
print("   stale_authority and weak_engagement both carry a cheap, specific action, and both")
print("   are pushed down to 0.15x. The model ranks by risk, not by actionability. Those two")
print("   things only partly coincide, and nothing in the ranking is trying to make them.")

residual = buckets["no_clear_defect"]
print(f"\n--- the honest residual ---")
print(f"no_clear_defect : {residual['queue_pages']} of {len(queue):,} delivered rows "
      f"({residual['queue_share']:.1%})")
print("-> for roughly a quarter of the queue this layer has no answer either. That number is")
print("   reported here rather than buried, and it is the ceiling on how much of the review")
print("   cycle can be planned in advance.")

eligible pages : 26,612   base rate: 0.611
pages matching >1 definition : 8,713 (33%) -> priority order is load-bearing

--- what the model actually selects, by archetype ---
             archetype                           action  eligible  elig %  in queue  queue %  skew  declining
           snippet_gap           rewrite_title_and_meta      3647  0.1370       254    0.254  1.85     0.6726
      thin_with_demand                     expand_depth        75  0.0028         0    0.000  0.00     0.5333
       stale_authority          refresh_facts_and_dates      1256  0.0472         7    0.007  0.15     0.5311
  deep_but_substantial   reassess_intent_or_consolidate      5424  0.2038       315    0.315  1.55     0.6792
       weak_engagement          review_intro_and_layout      3117  0.1171        18    0.018  0.15     0.5117
            low_demand no_content_edit_review_targeting      6047  0.2272       119    0.119  0.52     0.6306
insufficient_page_data             fix_the_record_first

**What the cell shows.**

The delivered queue is not a random slice of the corpus by archetype, and the way it differs is
the useful part.

**It concentrates.** `deep_but_substantial` (31.5%), `no_clear_defect` (28.5%) and `snippet_gap`
(25.4%) carry 85% of the queue between them. So a review cycle is, in practice, three jobs and not
eight: question the targeting on long pages that rank deep, rewrite listings on page-one pages
losing clicks, and diagnose the rest by hand.

**It de-selects two things it was never told to.** `low_demand` is 22.7% of eligible pages and
11.9% of the queue; `insufficient_page_data` is 10.8% of eligible pages and **0.2%** of the queue.
Nothing in the model knows that editing a no-demand page is wasted effort, or that a page with a
missing word count cannot be diagnosed. It pushed both down anyway, on visibility features alone.
That is a small independent check that the ranking behaves sensibly, and it is worth more than
another precision figure because it was not something the model was optimised for.

**But it also de-selects two things a reviewer would want.** `stale_authority` and
`weak_engagement` are pushed down to **0.15x** their corpus share, and both carry a cheap, specific
action — refresh the facts on a strong page, or fix an intro people bounce off. They are pushed
down because they are not what the model is scoring: the forest ranks on **risk of decline**, and
those two buckets decline at 0.531 and 0.512, near the 0.611 eligible base rate. Risk and
actionability are different quantities and this artefact optimises only the first. A cycle that
worked the queue top-down and nothing else would systematically never reach the easiest wins in the
corpus, which is a limitation of the design rather than a bug in the model, and it is carried into
section 2's limits.

**It leaves a quarter of the queue unexplained.** 285 of 1,000 delivered rows land in
`no_clear_defect`. That is the honest ceiling on advance planning: for 28.5% of the cycle, the
answer to *what do I do with this page* remains *look at it and decide*. Section 1's finding was
that the queue's **ordering** is not explainable; this is the same limitation showing up one level
down, in the **action**.

`thin_with_demand` gets **zero** rows in the top 1,000, from 75 eligible pages. Thin content is
essentially absent from what this model prioritises, which quietly contradicts a standard SEO
assumption and is worth carrying into the paper as a small negative result.

## 1c. Content decay, and what it does not license

*The card asks for the decay/refresh insight. The honest one runs against the folklore.*

The expected finding was that older content declines more, producing a refresh cadence: review a
page every N days. **The data says the opposite, and then says the cadence is not measurable.**

### The finding

Declining rate falls almost monotonically with content age: **0.673** in the 90–120 day band
against **0.439** past 470 days. Median trend goes from **−44.7%** to **−14.7%** across the same
range. Younger pages are the ones carrying the declining label.

The obvious objection is visibility: maybe young pages are just smaller. The cell below tests it by
re-running the breakdown inside each impression quartile, and **the pattern holds in all four.**
It is not a visibility artefact.

### Why I am not calling it a finding about content

Three mechanisms predict this same curve, and this data separates none of them:

1. **Post-launch settling.** The label compares two adjacent 30-day windows. A young page's *prior*
   window catches more of its launch ramp — median prev-30d share falls from 0.342 to 0.305 across
   the age range while last-30d share rises from 0.191 to 0.263. A page settling off a launch spike
   looks identical to a page declining.
2. **Survivorship.** ML-09 established the corpus is 100% active content. Pages that launched and
   died are gone, so the old bands contain only what stabilised.
3. **Feature degradation.** Median word count is **0** in both bands past 365 days — it is simply
   missing on most old pages. The model may rank old content lower partly because it can see less
   about it. This is the one mechanism that is fixable upstream, and it is a data-quality gradient
   rather than a content finding.

### The refresh cadence question, answered honestly: it cannot be

`days_since_last_update` takes **56 distinct values across 26,612 pages, and two of them cover
68%** of the corpus. It is very nearly a two-valued flag. The bands where it varies hold 128, 116
and 133 pages. **Refresh recency is not a continuous signal in this data, so no refresh cadence is
recommended.** Saying "review every 90 days" here would be inventing a number.

### What this does license, and it is a real recommendation

**Read the queue within an age band.** Ranked across the whole corpus, the youngest two bands take
**81%** of the delivered queue against 41% of the corpus, and pages older than 365 days take
**none of it at all** — zero rows, from 23% of the corpus. The queue is not a portfolio-wide review
tool. It is a young-content tool that has been presented as a portfolio-wide one, and a reviewer
who works it top-down will never see an old page.

In [3]:
# Section 1c - the age pattern, the confound test, and what the queue actually selects.
decay = playbook["content_decay"]

print("--- declining rate by content age ---")
age_rows = pd.DataFrame([
    {
        "age band (days)": band,
        "n": block["n"],
        "declining": block["declining_rate"],
        "median trend %": block["median_trend_pct"],
        "prev 30d share": block["median_prev_30d_share"],
        "last 30d share": block["median_last_30d_share"],
        "median words": block["median_word_count"],
    }
    for band, block in decay["by_content_age"].items()
])
print(age_rows.to_string(index=False))
print("-> an even split of a 90-day total would put 0.333 in each 30-day window.")
print("   Young pages run hot in the prior window and cold in the recent one: that is the")
print("   shape of a page settling after launch, which this label cannot tell from decay.")

print("\n--- the confound test: does it hold within each impression quartile? ---")
within = pd.DataFrame(decay["declining_rate_by_age_within_impression_quartile"]).T
print(within.to_string())
print(f"-> declining rate falls from youngest to oldest band in every quartile: "
      f"{decay['survives_the_visibility_confound']}")
assert decay["survives_the_visibility_confound"], \
    "the age pattern is only reportable because it survives the visibility confound"

print("\n--- can a refresh cadence be recommended from this? ---")
recency = decay["refresh_recency_is_nearly_two_valued"]
print(f"distinct values of days_since_last_update : {recency['distinct_values']}")
print(f"share held by the top two values          : {recency['top_two_values_share']:.1%}")
fresh = pd.DataFrame([
    {"days since update": band, "n": block["n"], "declining": block["declining_rate"]}
    for band, block in decay["by_days_since_last_update"].items()
])
print(fresh.to_string(index=False))
print("-> three of the five bands hold under 150 pages. No cadence is recommended.")

print("\n--- the recommendation this DOES support: the queue skews young ---")
skew = pd.DataFrame([
    {"age band (days)": band, "corpus %": block["corpus_share"],
     "queue %": block["queue_share"], "over-represented": block["over_representation"]}
    for band, block in decay["queue_age_skew"].items()
])
print(skew.to_string(index=False))
young_share = sum(b["queue_share"] for b in list(decay["queue_age_skew"].values())[:2])
old_share = sum(b["queue_share"] for b in list(decay["queue_age_skew"].values())[-2:])
old_corpus = sum(b["corpus_share"] for b in list(decay["queue_age_skew"].values())[-2:])
print(f"\nunder 180 days : {young_share:.1%} of the queue")
print(f"over 365 days  : {old_share:.1%} of the queue, from {old_corpus:.1%} of the corpus")
assert old_share == 0.0, "if old pages ever enter the queue, this recommendation must be re-derived"
print("-> a reviewer working this queue top-down will not see a single page older than a year.")
print("   Rank within age band if the cycle is meant to cover the portfolio.")

--- declining rate by content age ---
age band (days)    n  declining  median trend %  prev 30d share  last 30d share  median words
  (89.0, 120.0] 5033     0.6728          -44.70          0.3416          0.1912        2862.0
 (120.0, 180.0] 5858     0.7361          -48.40          0.3304          0.1705        2900.0
 (180.0, 270.0] 3934     0.6378          -36.05          0.3207          0.2078        3233.5
 (270.0, 365.0] 5680     0.5887          -30.05          0.3116          0.2192        1464.0
 (365.0, 470.0] 3617     0.4473          -15.00          0.3150          0.2723           0.0
   (470.0, inf] 2490     0.4390          -14.70          0.3051          0.2630           0.0
-> an even split of a 90-day total would put 0.333 in each 30-day window.
   Young pages run hot in the prior window and cold in the recent one: that is the
   shape of a page settling after launch, which this label cannot tell from decay.

--- the confound test: does it hold within each impression quar

**What the cell shows.**

The age pattern is real and it is not a visibility artefact: declining rate falls from the youngest
band to the oldest inside **every** impression quartile, and the assertion in the cell will fail
the notebook if that ever stops being true.

But the window-share columns show why it should not be read as a statement about content. Young
pages put **0.342** of their 90-day impressions in the prior window and only **0.191** in the
recent one. That is a page coming down off a launch ramp. The label compares exactly those two
windows and cannot distinguish it from a page that is genuinely losing ground.

The refresh side of the question closes cleanly and negatively: with 56 distinct values and 68% of
the corpus sitting on two of them, `days_since_last_update` cannot support a cadence. The three
bands where it actually varies hold 128, 116 and 133 pages. **This is a case where the useful
output is a documented refusal**, and it goes into the paper as one.

The queue skew is the recommendation that survives. Pages older than 365 days are **23% of the
corpus and 0% of the queue** — the assertion in the cell will break the notebook if that ever
changes, because the recommendation depends on it. Ranking across the whole corpus produces a
young-content queue. If a review cycle is supposed to cover the portfolio, it has to rank within
age band, and that is a change to how the artefact is used rather than a change to the model.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

**Who.** A content strategist or SEO reviewer with edit rights on the pages, working a prioritised
review cycle.

**For what.** Deciding the **order** in which to review pages for a possible refresh, when there is
more content than review capacity. That is the whole scope.

**What it replaces.** Reviewing in an arbitrary order, or by whoever asked most recently. Against the
0.542 base rate, working the top 1,000 in this order finds 778 declining pages instead of roughly
542 — about 236 fewer wasted reviews per thousand.

### What it does not do, stated as flatly as possible

| It does not | Why |
|---|---|
| **Predict that refreshing a page will recover traffic** | No refresh outcome exists anywhere in this data. The model ranks pages by whether impressions already fell, not by whether an edit would help. ML-09 traced this to the same overreach in the paper's Finding #4. |
| **Justify its own ordering row by row** | The two model reason codes fire on 100% of the queue. They explain the population, not the ranking within it. |
| **Apply to pages that already died** | The corpus is 100% active content (`impressions_90d > 0 and sessions_90d > 0`). The model has never seen a dead page and cannot rank one. |
| **Apply to new pages** | 3,388 pages have no prior 30-day window and are structurally excluded. A new page's risk is not a question this model can be asked. |
| **Compare clients** | Per-client declining rates run 0.000 to 0.937. A cross-client ranking reflects which client a page belongs to as much as the page's condition. Use it *within* a client. |
| **Support a causal or contractual claim** | Everything here is observational and decision-support. Nothing licenses "this page will decline" or "refreshing this recovers N%". |

### Where it stops being valid

- **If the label definition moves.** The label is `impressions_prev_30d → impressions_last_30d`
  change past **−20%**. ML-09 found the paper documents this as −10%; the data says −20%. A silent
  change to the threshold, the window, or the metric moves the label under the model with no error
  raised anywhere.
- **If the corpus stops being active-content-only.** The model was fitted on survivors. Feeding it a
  broader population is out-of-distribution use.
- **Beyond roughly 1,000 rows deep.** Not measured past that depth in this queue.
- **On a client with far more or fewer pages than the 32 here.** One client holds 12.3% of the
  corpus and 26.5% of the queue; fold-to-fold variation of ±0.054 is driven largely by that
  imbalance.

In [4]:
# Section 2 - the limits, measured rather than asserted.
concentration = playbook["concentration"]
print("--- who is actually in this queue ---")
print(f"clients represented        : {concentration['clients_in_queue']} of {corpus['clients']}")
print(f"largest client, queue share: {concentration['largest_client_share_of_queue']:.1%}")
print(f"largest client, corpus share: {concentration['largest_client_share_of_corpus']:.1%}")
over = concentration["largest_client_share_of_queue"] / concentration["largest_client_share_of_corpus"]
print(f"  -> over-represented {over:.1f}x relative to its share of the corpus")
print(f"top 5 clients              : {concentration['top_5_clients_share_of_queue']:.1%} of the queue")
print("-> below the 50% single-client trigger in section 4, but the reason the playbook")
print("   says 'use it within a client' rather than across clients.")

# The value proposition, computed rather than claimed.
depth = decision["queue_depth"]
found = playbook["queue_quality"][f"p@{depth}"]["declining_pages_found"]
expected_by_chance = corpus["base_rate"] * depth
print(f"\n--- what the ordering is worth at depth {depth} ---")
print(f"declining pages found      : {found}")
print(f"expected in a random {depth}   : {expected_by_chance:.0f}")
print(f"wasted reviews avoided     : {found - expected_by_chance:.0f}")
print(f"lift over base rate        : {playbook['queue_quality'][f'p@{depth}']['lift_vs_base_rate']}x")

# The boundary claims, asserted so a future edit cannot quietly break them.
assert corpus["held_out_declining_count"] == 0, \
    "pages with no prior window should be structurally unlabelable as declining"
assert not playbook["exports"]["label_columns_in_queue"], \
    "no label column may reach a queue a human works from"
assert playbook["explainability"]["delivered_rows_unexplained"] == 0, \
    "every delivered row must carry a justification"
print("\n[OK] three boundary conditions hold: no label columns in the delivered CSV,")
print("     no structurally-unlabelable pages ranked, no unexplained rows.")

--- who is actually in this queue ---
clients represented        : 18 of 32
largest client, queue share: 26.5%
largest client, corpus share: 12.3%
  -> over-represented 2.2x relative to its share of the corpus
top 5 clients              : 65.9% of the queue
-> below the 50% single-client trigger in section 4, but the reason the playbook
   says 'use it within a client' rather than across clients.

--- what the ordering is worth at depth 1000 ---
declining pages found      : 778
expected in a random 1000   : 542
wasted reviews avoided     : 236
lift over base rate        : 1.4352x

[OK] three boundary conditions hold: no label columns in the delivered CSV,
     no structurally-unlabelable pages ranked, no unexplained rows.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### The review gate — four checks, in order, before any page is edited

The queue produces **candidates**, not instructions. Each check below can be done in under a minute
and each one exists because something in this repo showed it was needed.

1. **Is the decline real, or is it the measurement?** Open the page's own impression series. The
   label is one 30-day window against the previous one, cut at −20%. A seasonal dip, a SERP feature
   change, or a tracking gap all read as "declining" to this model. *Why:* the label is a difference
   of two noisy counts, and ML-09 showed 3,388 pages were classified purely by whether a prior
   window existed at all.
2. **Does the reason code match what you see?** The `explanation_source` column says whether the
   justification came from the ML-07 rule or from the model. If it says `model`, the rule found
   nothing wrong with the page — read that as "the model noticed a pattern", not as a diagnosis.
   *Why:* 444 of 1,000 rows are in exactly that position.
3. **Is a refresh the right instrument?** A page can be declining because the topic is dead, because
   a competitor published something better, or because it was cannibalised by another page on the
   same site. Only one of those is fixed by refreshing. *Why:* the model has no feature for any of
   them.
4. **Would you defend this edit to the client?** If the only justification is "it was ranked 12th",
   that is not a justification — the within-queue ordering is not explainable (section 1).

### The no-go list

**Never automated, under any confidence score:**

- **Publishing an edit.** Nothing in this system writes to a page. Full stop.
- **Deleting, de-indexing, or redirecting a page.** The model was never trained to distinguish "not
  worth refreshing" from "should not exist", and those failure costs are not symmetric.
- **Client-facing performance promises.** No recovery outcome was observed, so no recovery can be
  promised.
- **Budget or headcount allocation between clients.** Per-client base rates run 0.000 to 0.937.
- **Automated bulk regeneration of flagged pages.** Refresh-by-machine on a queue with 22% false
  positives at depth 1,000 damages the pages that were fine.

**Never done without a second person:**

- Acting on any page where the reason codes and the visible evidence disagree.
- Changing the `−20%` label threshold, the queue depth, or the shipped model. Each one silently
  changes what every downstream number means.
- Reusing the queue for a purpose not in section 2's intended-use list.

### The failure mode I am most worried about

Not a wrong ranking — a **confident-looking wrong ranking**. The queue exports a `confidence`
column, and it is easy to read that as calibrated probability. It is not: the bands are tertiles of
this queue's own score distribution, so "high confidence" means *high relative to the other 999 rows
in front of you* and nothing more. The forest's probabilities were never calibrated and the playbook
makes no calibration claim. The cell below shows what those bands actually correspond to.

In [5]:
# Section 3 - what 'confidence' does and does not mean.
bands = queue.groupby("confidence").agg(
    rows=("queue_rank", "size"),
    min_score=("model_score", "min"),
    max_score=("model_score", "max"),
    mean_rank=("queue_rank", "mean"),
).sort_values("min_score", ascending=False).round(4)
print("--- the confidence bands are tertiles of THIS queue, not calibrated probabilities ---")
print(bands.to_string())
print(f"\nscore range across the whole delivered queue: "
      f"{queue['model_score'].min():.4f} to {queue['model_score'].max():.4f}")
print("-> 'low confidence' here still means a score above the threshold that got it into the")
print("   top 1,000 of 26,612. It does not mean the page is probably fine.")

print("\n--- where the justification comes from, by band ---")
print(pd.crosstab(queue["confidence"], queue["explanation_source"]).to_string())

# Review load, so the cycle can actually be planned.
print("\n--- review load by action, at 3 minutes per page ---")
load = queue["suggested_action"].value_counts().to_frame("pages")
load["minutes"] = load["pages"] * 3
load["hours"] = (load["minutes"] / 60).round(1)
print(load.to_string())
print(f"\ntotal: {load['hours'].sum():.1f} hours to work the full {len(queue):,}-page queue")

--- the confidence bands are tertiles of THIS queue, not calibrated probabilities ---
            rows  min_score  max_score  mean_rank
confidence                                       
high         333     0.8230     0.9257      167.0
medium       333     0.8042     0.8230      500.0
low          334     0.7923     0.8041      833.5

score range across the whole delivered queue: 0.7923 to 0.9257
-> 'low confidence' here still means a score above the threshold that got it into the
   top 1,000 of 26,612. It does not mean the page is probably fine.

--- where the justification comes from, by band ---
explanation_source  model  rule+model
confidence                           
high                  143         190
low                   155         179
medium                146         187

--- review load by action, at 3 minutes per page ---
                            pages  minutes  hours
suggested_action                                 
review_visibility_trend       444     1332   22.2

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### The principle behind every threshold

ML-09 measured a fold-to-fold spread of **±0.054** on precision@1000 across held-out clients. That
is the scale of ordinary variation in this corpus. **A trigger inside that band is a false alarm
generator**, so every threshold below sits outside it, and each one is anchored to a number this
repo actually measured rather than to a round figure.

One threshold was deliberately loosened after measuring it. My first draft set the explainability
trigger at 80% rule coverage — but measured coverage on day one is 55.6%. A monitor that is already
breached the moment it ships is not a monitor, it is noise. It now sits at 45%, below today's
value, and the thing that must never move (total coverage = 100%) is enforced by an assertion in
`action_playbook.py` instead of watched by a human.

### The triggers

In [6]:
# Section 4 - the trigger table, with every baseline traced to a measurement.
triggers = playbook["monitoring_triggers"]
print(triggers["principle"])

for t in triggers["triggers"]:
    print(f"\n{'=' * 78}\n{t['name'].upper()}")
    print(f"  watch     : {t['watch']}")
    if "baseline" in t:
        print(f"  baseline  : {t['baseline']}")
    for key in ("trigger_below", "trigger_above", "trigger_outside", "trigger_on"):
        if key in t:
            print(f"  {key:<10}: {t[key]}")
    print(f"  why       : {t['rationale']}")
    print(f"  action    : {t['action']}")

print(f"\n{'=' * 78}\ncadence")
for when, names in triggers["review_cadence"].items():
    print(f"  {when:<24} {', '.join(names)}")

ML-09 measured a fold-to-fold spread of +/-0.054 on precision@1000. Anything inside that band is fold noise; the triggers below sit outside it.

QUEUE_PRECISION_DECAY
  watch     : share of the delivered top-1000 that a reviewer confirms as declining
  baseline  : 0.778
  trigger_below: 0.67
  why       : two fold-standard-deviations below what this queue actually delivers. The +/-0.054 comes from ML-09's precision@1000 spread across held-out clients and is used only as the noise scale -- the 0.726 mean it came from is NOT this number and the two are not comparable (see the notebook's section 1 note on denominators)
  action    : retrain; if it does not recover, fall back to the ML-07 rule

BASE_RATE_SHIFT
  watch     : corpus-wide declining rate
  baseline  : 0.5421
  trigger_outside: [0.442, 0.642]
  why       : +/-0.10 around the 0.542 the model was fitted at; the queue's lift is quoted against that base rate and stops meaning the same thing if it moves
  action    : re-quote every 

### What is deliberately not monitored

- **Model score drift on its own.** The forest's probabilities are uncalibrated, so a shift in the
  score distribution is not interpretable without a label to check it against. `queue_precision_decay`
  covers the same ground with something measurable.
- **Feature importance stability.** Interesting for a report, but importance moving is not by itself
  evidence the queue got worse, and treating it as a trigger invites retraining for no reason.
- **Total explanation coverage.** It must be 100% and it is asserted in code. A condition that must
  never be false belongs in an assertion, not on a dashboard.

### The retrain decision, in one line

Retrain on `queue_precision_decay` or `base_rate_shift`. **Stop and re-derive** on
`label_definition_change` — that one is not a retrain, it is a rebuild, because every number in
ML-07 through ML-10 was computed against the −20% cut.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Two artefacts, and the split follows `work/README.md` rule 2 rather than convenience:

| File | Committed? | Why |
|---|---|---|
| `work/outputs/action_playbook_metrics.json` | **yes** | Every number quoted in this notebook and in the capstone traces to it. Metrics JSONs are the receipts. |
| `work/outputs/refresh_action_queue.csv` | no — gitignored | It is derived data, it regenerates from `action_playbook.py` in about a minute, and CI fails the build if a dataset CSV is committed anywhere in the repo. |

The figure below is written as SVG rather than PNG so it stays diff-readable in git and needs no
extra dependency in Colab.

In [7]:
# Section 5 - export the figure and confirm the receipts the capstone will read.
FIG_DIR = ROOT / "work" / "figures"
FIG_DIR.mkdir(exist_ok=True)
FIG_PATH = FIG_DIR / "queue_precision_by_depth.svg"

quality = playbook["queue_quality"]
depths = list(quality)
precisions = [quality[d]["precision"] for d in depths]
base = corpus["base_rate"]

W, H, PAD = 640, 300, 52
plot_w, plot_h = W - 2 * PAD, H - 2 * PAD
bar_w = plot_w / (len(depths) * 1.6)
y_max = 1.0

parts = [
    f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" height="{H}">',
    '<style>text{font-family:system-ui,sans-serif;font-size:12px;fill:#333}'
    '.t{font-size:14px;font-weight:600}.s{font-size:11px;fill:#777}</style>',
    f'<rect width="{W}" height="{H}" fill="white"/>',
    f'<text class="t" x="{PAD}" y="24">Queue precision by review depth</text>',
    f'<text class="s" x="{PAD}" y="40">out-of-fold random forest, {corpus["eligible_for_queue"]:,} '
    f'eligible pages, base rate {base:.3f}</text>',
]
base_y = PAD + plot_h - (base / y_max) * plot_h
parts.append(
    f'<line x1="{PAD}" y1="{base_y:.1f}" x2="{PAD + plot_w}" y2="{base_y:.1f}" '
    'stroke="#c33" stroke-width="1" stroke-dasharray="4 3"/>'
)
parts.append(f'<text class="s" x="{PAD + plot_w - 96}" y="{base_y - 5:.1f}" fill="#c33">'
             f'base rate {base:.3f}</text>')
for i, (d, p) in enumerate(zip(depths, precisions)):
    x = PAD + i * (plot_w / len(depths)) + (plot_w / len(depths) - bar_w) / 2
    h = (p / y_max) * plot_h
    y = PAD + plot_h - h
    parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{h:.1f}" fill="#3b6ea5"/>')
    parts.append(f'<text x="{x + bar_w / 2:.1f}" y="{y - 6:.1f}" text-anchor="middle">{p:.3f}</text>')
    parts.append(f'<text x="{x + bar_w / 2:.1f}" y="{PAD + plot_h + 18:.1f}" '
                 f'text-anchor="middle">{d}</text>')
parts.append(f'<line x1="{PAD}" y1="{PAD + plot_h}" x2="{PAD + plot_w}" y2="{PAD + plot_h}" '
             'stroke="#999"/>')
parts.append('</svg>')
FIG_PATH.write_text("\n".join(parts), encoding="utf-8")

print(f"figure  -> {FIG_PATH.relative_to(ROOT)}")
print(f"metrics -> {playbook['exports']['metrics_json']}  (committed)")
print(f"queue   -> {playbook['exports']['queue_csv']}  (gitignored, regenerates)")
print(f"          {len(queue):,} rows x {len(queue.columns)} columns")

print("\nwhat the capstone will read out of the metrics receipt:")
for key in ("operating_decision", "corpus", "queue_quality", "explainability",
            "archetypes", "content_decay",
            "concentration", "monitoring_triggers", "exports"):
    print(f"  playbook['{key}']")
    assert key in playbook, f"the capstone expects playbook['{key}'] and it is missing"

# The recommendations section of the paper is built from these two, so name the
# exact paths rather than making next week re-derive them.
print("\nthe recommendations section builds on:")
print(f"  playbook['archetypes']['buckets']            "
      f"-> {len(playbook['archetypes']['buckets'])} archetype -> action rows")
print(f"  playbook['content_decay']['queue_age_skew']  "
      f"-> the young-content skew, {len(playbook['content_decay']['queue_age_skew'])} age bands")
print(f"  queue columns page_archetype / archetype_action "
      f"-> per-row action for all {len(queue):,} delivered rows")

assert FIG_PATH.exists() and OUT_JSON.exists(), "the paper's inputs must both exist"
assert {"page_archetype", "archetype_action"} <= set(queue.columns), \
    "the delivered queue must carry its own action column"
print("\n[OK] both paper inputs written.")

figure  -> work\figures\queue_precision_by_depth.svg
metrics -> work/outputs/action_playbook_metrics.json  (committed)
queue   -> work/outputs/refresh_action_queue.csv  (gitignored, regenerates)
          1,000 rows x 29 columns

what the capstone will read out of the metrics receipt:
  playbook['operating_decision']
  playbook['corpus']
  playbook['queue_quality']
  playbook['explainability']
  playbook['archetypes']
  playbook['content_decay']
  playbook['concentration']
  playbook['monitoring_triggers']
  playbook['exports']

the recommendations section builds on:
  playbook['archetypes']['buckets']            -> 8 archetype -> action rows
  playbook['content_decay']['queue_age_skew']  -> the young-content skew, 6 age bands
  queue columns page_archetype / archetype_action -> per-row action for all 1,000 delivered rows

[OK] both paper inputs written.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — clients appear only as anonymised `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Carried into the capstone (ML-11):**

1. **Shipped:** random forest, out-of-fold scored, top 1,000 of 26,612 eligible pages, precision
   0.778 at that depth — a figure about this artefact, not a generalisation claim. The
   generalisation claim remains ML-09's **0.726 ± 0.054**.
2. **The explainability finding is a result, not a caveat.** The readable rule and the learned model
   disagree about what matters, and measuring that disagreement (55.6% coverage) is more useful than
   either scorer alone.
3. **The queue orders review; it does not predict recovery.** No refresh outcome exists in this data.
4. **3,388 pages are structurally unrankable** and are excluded rather than ranked low.
5. **Eight page archetypes carry the recommendations section.** Three of them —
   `deep_but_substantial`, `no_clear_defect`, `snippet_gap` — are 85% of the delivered queue, so a
   review cycle is three jobs, not eight. `thin_with_demand` takes **zero** of the top 1,000, which
   is a small negative result against a standard SEO assumption and worth stating as one.
6. **The model ranks by risk, not by actionability.** It correctly buries pages it cannot diagnose
   (`insufficient_page_data`, 0.02x) and pages with no demand (`low_demand`, 0.52x), neither of
   which it was told to do. It also buries `stale_authority` and `weak_engagement` at 0.15x, and
   those are cheap wins. Working the queue top-down and nothing else never reaches them.
7. **The decay finding runs against the folklore and survives its confound.** Declining rate falls
   from **0.673** (90–120 days) to **0.439** (470+ days), inside *every* impression quartile. It is
   still not a claim about content: post-launch settling, survivorship, and missing word counts on
   old pages all predict the same curve and this data separates none of them.
8. **No refresh cadence is recommended, and the refusal is the result.** `days_since_last_update`
   has 56 distinct values with 68% of the corpus on two of them. It is not a continuous signal.
9. **The queue is a young-content tool.** 81% of it is under 180 days old and pages past 365 days
   take **0%** of it, from 23% of the corpus. Rank within age band if a cycle must cover the
   portfolio — a change to how the artefact is used, not to the model.